# Excel explorer

Select a source Excel file from `data/` to inspect its sheets and preview the first sheet.

In [1]:
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Locate the repository root so this works from Jupyter's chosen cwd."""
    for directory in (start, *start.parents):
        if (directory / 'data').is_dir():
            return directory
    raise FileNotFoundError("Could not find the repository's data directory.")


project_root = find_project_root(Path.cwd().resolve())
excel_files = sorted((project_root / 'data').glob('*.xlsx'))

if not excel_files:
    raise FileNotFoundError(f"No .xlsx files found in {project_root / 'data'}")

file_picker = widgets.Dropdown(
    options=[(file.name, file) for file in excel_files],
    description='Excel file:',
    layout=widgets.Layout(width='700px'),
)
sheet_picker = widgets.Dropdown(description='Sheet:', layout=widgets.Layout(width='400px'))
output = widgets.Output()


In [2]:
def show_preview(*_):
    """Load the selected sheet and display a small, safe preview."""
    selected_file = file_picker.value
    with output:
        output.clear_output(wait=True)
        try:
            frame = pd.read_excel(selected_file, sheet_name=sheet_picker.value, nrows=10)
            print(f"File: {selected_file.name}")
            print(f"Sheet: {sheet_picker.value} | Showing first {len(frame)} rows")
            display(frame)
        except Exception as error:
            print(f"Could not read the selected sheet: {error}")


def update_sheets(*_):
    """Update the sheet dropdown after choosing another workbook."""
    try:
        workbook = pd.ExcelFile(file_picker.value)
        sheet_picker.options = workbook.sheet_names
        sheet_picker.value = workbook.sheet_names[0]
    except Exception as error:
        with output:
            output.clear_output(wait=True)
            print(f"Could not open the selected workbook: {error}")


file_picker.observe(update_sheets, names='value')
sheet_picker.observe(show_preview, names='value')
update_sheets()
display(widgets.VBox([file_picker, sheet_picker, output]))
